In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_6 import BioJepa, BioJepaConfig
from dataloader_v0_6 import PretrainLoader, AlignmentLoader, TrainingLoader
from training_v0_6 import create_model, load_feature_banks, run_pretraining, run_alignment, run_full_training, train_linear_decoder, maybe_compile
from config_v0_6 import PretrainConfig, AlignmentConfig, FullTrainingConfig, DecoderConfig, DataConfig
from evals.evals import EvalContext, run_pretraining_evals, run_alignment_evals, run_full_model_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('/home/ubuntu/data/v0_6')
ref_root = Path('/home/ubuntu/data/reference_data')

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoints',
    ref_dir = ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


In [3]:
# Model architecture
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=2,
    heads=2,
    embed_dim=8,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.6,
    gaussian_scale=2.0,
    film_linear_multiple=1.0,
    sim_coeff=25.0,
    std_coeff=25.0,
    cov_coeff=1.0,
    pert_latent_dim= 8,
    pert_mode_dim= 8,
)

# Training configs
pt_cfg = PretrainConfig(epochs=1, lr=1e-3, batch_size=128) 
align_cfg = AlignmentConfig(epochs=100, lr=4e-3, batch_size=32)
full_cfg = FullTrainingConfig(epochs=1, predictor_lr=1e-3, batch_size=32) 
decoder_cfg = DecoderConfig(epochs=1, lr=1e-3, batch_size=16) 

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)
seq_banks, target_bank = load_feature_banks(data_cfg, device)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters() if p.requires_grad):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters() if p.requires_grad):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters() if p.requires_grad):,}')

Loaded DNA embeddings: torch.Size([11643, 1536])
Loaded chemical embeddings: torch.Size([188, 1536])
Loaded target embeddings: torch.Size([9975, 320])
Student/Teacher: 84,074
ACpredictor: 84,864
PerturbationComposer: 26,152


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_full_final.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [6]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': pt_cfg.batch_size, 'seed': SEED
})
pt_eval_results = run_pretraining_evals(eval_ctx)

save_report(pt_eval_results, data_cfg.eval_results_dir / 'pretraining_eval_report.json')
pt_eval_results

Using cuda
found 121 shards for split test


batch_invariance: Extracting embeddings: 100%|███████████████████████| 2420/2420 [00:52<00:00, 45.87it/s]


Training classifiers...
batch_invariance: Batch=0.0073 (2.8x), Pert=0.0177 (19.2x)
batch_invariance summary: global_ratio=2.411, within_dataset_macro_ratio=2.218
Loading KEGG_2021_Human...
  320 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
gene_embedding_pathways: KEGG sil=-0.2286, Reactome sil=-0.2079
essential_gene_prediction: Pearson=0.1100, AUROC=0.5859
found 121 shards for split test


cell_type_probing: Extracting embeddings: 100%|██████████████████████| 2420/2420 [00:51<00:00, 47.06it/s]


Training cell type classifier...
cell_type_probing: Accuracy=0.8050 (2.4x chance), Macro F1=0.4603
found 121 shards for split test


...

embedding_consistency: Extracting embeddings: 100%|██████████████████| 2420/2420 [00:51<00:00, 46.74it/s]


embedding_consistency: Computing intra-distances for 1084 perturbations...
embedding_consistency: Computing 5000 inter-distances...
embedding_consistency: Intra=2.6822, Inter=2.2091, Ratio=0.82x
embedding_consistency: Computing for dataset adamson...
embedding_consistency: Computing for dataset k562e_raw...
embedding_consistency: Computing for dataset k562gw...
embedding_consistency: Computing for dataset norman...
embedding_consistency: Computing for dataset sciplex...
found 121 shards for split test


latent_space_health: Extracting embeddings: 100%|████████████████████| 2420/2420 [00:51<00:00, 46.99it/s]

latent_space_health: Eff_dim_90=3/8, Mean_var=0.6646, Isotropy=0.000108
Saved report to /home/ubuntu/data/v0_6/eval_results/pretraining_eval_report.json


{'batch_invariance': {'config': {'samples': 309760,
   'embedding_dim': 8,
   'num_batches': 376,
   'num_perturbations': 1084},
  'batch_classifier': {'accuracy': 0.007344395661157025,
   'chance': 0.0026595744680851063,
   'above_chance_ratio': 2.7614927685950414},
  'perturbation_classifier': {'accuracy': 0.017707257231404958,
   'chance': 0.0009225092250922509,
   'above_chance_ratio': 19.194666838842974},
  'invariance_ratio': 2.410989010989011,
  'by_dataset': {'k562e_raw': {'config': {'samples': 49066,
     'embedding_dim': 8,
     'num_batches': 48,
     'num_perturbations': 286},
    'batch_classifier': {'accuracy': 0.02394538414509884,
     'chance': 0.020833333333333332,
     'above_chance_ratio': 1.1493784389647443},
    'perturbation_classifier': {'accuracy': 0.012635011208477685,
     'chance': 0.0034965034965034965,
     'above_chance_ratio': 3.6136132056246177},
    'invariance_ratio': 0.5276595744680851},
   'k562gw': {'config': {'samples': 178474,
     'embedding_dim'

In [7]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [9]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': align_cfg.batch_size, 'seed': SEED
})
align_eval_results = run_alignment_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'alignment_eval_report.json')
align_eval_results

Using cuda
Loaded 10797 v0.6 alignment pairs
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Encoded DNA sequences: torch.Size([11643, 8])
Encoded chemical sequences: torch.Size([188, 8])
Loaded target bank: torch.Size([9975, 320])
Encoded protein targets: torch.Size([9975, 8])
seq_to_target_retrieval: dna_mrr=0.0010
cross_modality_target_consistency: Within=0.9873, Between=0.9848, Ratio=1.00x
seq_target_gap_analysis: dna_gap=5.14
paired_alignment_quality: dna_sim=0.1079
mode_sensitivity: Classification_acc=0.0048 (0.0x chance)
fusion_quality: Fused_var=5.4531, Seq_var=5.2794, Target_var=0.0380
missing_data_robustness: Fused_MRR=0.0012, Seq_only=0.0011, Target_only=0.9975
found 121 shards for split test


multi_pert_alignment: Scanning for multi-pert: 100%|██████████████████| 500/500 [00:03<00:00, 162.91it/s]


Loaded 27264 HGNC gene family annotations
target_family_probing: seq_only=0.0240, target_only=0.0696, fused=0.0616
Saved report to /home/ubuntu/data/v0_6/eval_results/alignment_eval_report.json


{'seq_to_target_retrieval': {'config': {'n_targets': 9975},
  'by_modality': {'dna': {'mrr': 0.0010070404655683988,
    'median_rank': 4860.0,
    'mean_rank': 4881.335245978741,
    'n_queries': 10631,
    'n_targets': 9975,
    'recall_at_k': {'1': 9.406452826639074e-05,
     '5': 0.0005643871695983444,
     '10': 0.0009406452826639074,
     '20': 0.0020694196218605963,
     '50': 0.0056438716959834444}}}},
 'cross_modality_target_consistency': {'config': {'n_valid_targets': 915,
   'n_within_pairs': 1452,
   'n_between_pairs': 5000},
  'metrics': {'within_target_sim': 0.9872722625732422,
   'between_target_sim': 0.9847651924967766,
   'consistency_ratio': 1.002545855697955}},
 'seq_target_gap_analysis': {'target_variance': 0.3349124491214752,
  'n_targets': 9975,
  'dna': {'seq_variance': 46.353614807128906,
   'centroid_distance': 41.52951431274414,
   'mean_within_seq': 8.08043066948314,
   'mean_seq_to_target': 41.503700256347656,
   'gap_ratio': 5.136322796889044,
   'n_sequence

In [10]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [6]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
decoder.load_state_dict(decoder_sd)

<All keys matched successfully>

In [7]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': full_cfg.batch_size, 'seed': SEED, 'test_total_examples': 20000,
})
full_eval_results = run_full_model_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'full_model_eval_report.json')
full_eval_results

Using cuda
found 121 shards for split test


Running test inference:   0%|                                                    | 0/625 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


Running test inference: 100%|██████████████████████████████████████████| 625/625 [01:20<00:00,  7.75it/s]


Aggregated 1062 perturbations, 20000 samples, 8 shards
  k562e_raw: 280 perturbations, 2560 samples
  k562gw: 1044 perturbations, 12320 samples
  sciplex: 14 perturbations, 5120 samples
Cached test inference to /home/ubuntu/data/v0_6/test_inference_cache (8 shards)
expression_prediction: Pearson=0.9256, R2=0.8259, Centroid_acc=0.0038
gene_level_analysis: Dir_acc=0.8303, Top50_acc=0.2223


perturbation_retrieval (dna): 100%|████████████████████████████████████| 100/100 [03:27<00:00,  2.08s/it]


perturbation_retrieval (dna): MRR=0.0006


perturbation_retrieval (chemical): 100%|█████████████████████████████████| 13/13 [00:00<00:00, 31.51it/s]


perturbation_retrieval (chemical): MRR=0.0220
uncertainty_calibration: ECE=0.1565, Monotonicity=66.67%
Loading KEGG_2021_Human...
  320 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
Encoded DNA sequences: torch.Size([11643, 8])
Encoded chemical sequences: torch.Size([188, 8])
Encoded protein targets: torch.Size([9975, 8])
action_vector_pathways DNA: sil=-0.3706042170524597
Loaded 10797 v0.6 alignment pairs
moa_matching: Within=0.7035, Between=0.5789, Ratio=1.215x
dose_response: monotonicity=52.38%, spearman=0.0484
Saved report to /home/ubuntu/data/v0_6/eval_results/full_model_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1062,
   'genes': 10000,
   'test_samples': 20000},
  'sample_level': {'mse': 0.2858598836325109,
   'pearson_r_top20': 0.6312723825142719},
  'perturbation_level': {'r2_all_genes': {'mean': 0.8258523734500404,
    'median': 0.8618157207965851},
   'r2_top50_degs': {'mean': -0.16609899438707168,
    'median': -0.01764625310897827},
   'mse': {'mean': 0.038211390376091, 'median': 0.030360359698534012},
   'pearson_all_genes': {'mean': 0.9256466835905603,
    'median': 0.9446991682052612},
   'pearson_delta_all_genes': {'mean': 0.3243802959128351,
    'median': 0.3354436755180359},
   'pearson_top50_degs': {'mean': 0.5488087681653557,
    'median': 0.5842018723487854}},
  'centroid_accuracy': 0.003766478342749529,
  'vs_baseline': {'beat_rate': 0.2674199623352166, 'n_evaluated': 1062},
  'severity': {'pearson_r': 0.7234843969345093,
   'spearman_r': 0.5434964650361969},
  'error_by_magnitude': {'0-0.25': {'mae': 0.12584874033927

In [8]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()